In [ ]:
using Jens, Cosmology, Statistics
using Jens:LensPlots as jp

In [ ]:
obs = LensFITS.read_fits("../img/HE0435−1223.fits";
    noise = :gauss_poiss,
    noise_method = :clipped,
    noise_kwargs = (clip_sigma=3.0, max_iter=5),
    center = (976, 844), radius=32
    )

In [ ]:
println("exposure_time = ", obs.exposure_time, " s")
println("noise type    = ", typeof(obs.noise))
println("data shape    = ", size(obs.data))
println("pix_size      = ", obs.grid.pix_size, " arcsec/pix")

c = jp.LensCanvas(xlims=(-3, 3), ylims=(-3, 3))
jp.PlotPlane!(c, obs.data; title="HE0435-1223 (F160W)")
display(c)

In [ ]:
cosmo = Cosmology.FlatLCDM(0.6736, 0.3153, 0.0, 0.0)

# Lens: SIE at z_lens ~ 0.45
lens = LensBase.SingleModel(LensModel.SIE;
    theta_E = 1.2, e1 = 0.15, e2 = -0.05,
    xcentre = 0.05, ycentre = -0.03)
lp = LensGenerator.LensedPlane(lens; z_lens=0.45, cosmology=cosmo)

# Source: spherical Sersic at z_source ~ 1.69
src = LightModel.ExtendedSource(LightModel.SersicLight.SersicSpheric;
    amp = 0.3, Rsersic = 0.2, n = 2.0,
    xcentre = 0.02, ycentre = 0.01)
sp = LensGenerator.LightPlane(src; z=1.69)

# Attach PSF (WFC3/IR F160W, FWHM ~ 0.13 arcsec)
obs = LensObservation.with_psf(obs, LensPSF.GaussianPSF(fwhm=0.13))

# Build ForwardModel
sys = LensSystem.ForwardModel(;
    lens_plane   = lp,
    source_plane = sp,
    grid         = obs.grid,
    psf          = obs.psf,
    mask         = obs.mask)

img = LensSystem.render(sys)
logp = LensSystem.masked_logp(sys, obs.data, obs.noise, obs.mask)

println("log P    = ", round(logp; digits=1))
println("chi2/dof = ", round(-2*logp / sum(obs.mask); digits=2), " (hand-guessed params)")

In [ ]:
c1 = jp.LensCanvas(xlims=(-3, 3), ylims=(-3, 3))
jp.PlotPlane!(c1, obs.data; title="Data")
display(c1)

c2 = jp.LensCanvas(xlims=(-3, 3), ylims=(-3, 3))
jp.PlotPlane!(c2, img; title="Model (initial)")
display(c2)

c3 = jp.LensCanvas(xlims=(-3, 3), ylims=(-3, 3))
jp.PlotPlane!(c3, obs.data .- img; title="Residual")
display(c3)

In [ ]:
# MCMC: fit SIE(theta_E, e1, e2) + Sersic(amp, Rsersic, n)
# obs, cosmo are captured from previous cells

function logp_fn(p)
    theta_E, e1, e2, amp, Rsersic, n = p

    lens = LensBase.SingleModel(LensModel.SIE;
        theta_E=theta_E, e1=e1, e2=e2,
        xcentre=0.05, ycentre=-0.03)
    lp = LensGenerator.LensedPlane(lens; z_lens=0.45, cosmology=cosmo)

    src = LightModel.ExtendedSource(LightModel.SersicLight.SersicSpheric;
        amp=amp, Rsersic=Rsersic, n=n,
        xcentre=0.02, ycentre=0.01)
    sp = LensGenerator.LightPlane(src; z=1.69)

    sys = LensSystem.ForwardModel(;
        lens_plane=lp, source_plane=sp,
        grid=obs.grid, psf=obs.psf, mask=obs.mask)

    return LensSystem.masked_logp(sys, obs.data, obs.noise, obs.mask)
end

lower = [0.5, -0.4, -0.4, 0.01, 0.01, 0.5]
upper = [2.5,  0.4,  0.4, 1.0,  1.0,  6.0]

println("Running MCMC (6 params, 8 starts, 2000 steps)...")
result = LensMH.lens_mh_multistart(logp_fn, lower, upper;
    n_starts=8, n=2000, seed=42)

m, s = LensMH.chain_stats(result; burn=500)
println("\nPosterior (mean +- std):")
println("  theta_E  = ", round(m[1]; digits=3), " +- ", round(s[1]; digits=3))
println("  e1       = ", round(m[2]; digits=3), " +- ", round(s[2]; digits=3))
println("  e2       = ", round(m[3]; digits=3), " +- ", round(s[3]; digits=3))
println("  amp      = ", round(m[4]; digits=3), " +- ", round(s[4]; digits=3))
println("  Rsersic  = ", round(m[5]; digits=3), " +- ", round(s[5]; digits=3))
println("  n_sersic = ", round(m[6]; digits=3), " +- ", round(s[6]; digits=3))
println("  accepted = ", result.accepted, "/ 2000")

In [ ]:
# Render best-fit model
p_best = m  # posterior mean

lens_best = LensBase.SingleModel(LensModel.SIE;
    theta_E=p_best[1], e1=p_best[2], e2=p_best[3],
    xcentre=0.05, ycentre=-0.03)
lp_best = LensGenerator.LensedPlane(lens_best; z_lens=0.45, cosmology=cosmo)

src_best = LightModel.ExtendedSource(LightModel.SersicLight.SersicSpheric;
    amp=p_best[4], Rsersic=p_best[5], n=p_best[6],
    xcentre=0.02, ycentre=0.01)
sp_best = LensGenerator.LightPlane(src_best; z=1.69)

sys_best = LensSystem.ForwardModel(;
    lens_plane=lp_best, source_plane=sp_best,
    grid=obs.grid, psf=obs.psf, mask=obs.mask)

img_best = LensSystem.render(sys_best)
logp_best = LensSystem.masked_logp(sys_best, obs.data, obs.noise, obs.mask)

println("Best-fit log P    = ", round(logp_best; digits=1))
println("Best-fit chi2/dof = ", round(-2*logp_best / sum(obs.mask); digits=2))

cb = jp.LensCanvas(xlims=(-3, 3), ylims=(-3, 3))
jp.PlotPlane!(cb, obs.data .- img_best; title="Best-fit Residual")
display(cb)

## Summary

Full pipeline: FITS -> Observation -> ForwardModel -> MCMC -> posterior

1. **`read_fits`** — read HST/WFC3 drizzled data into `Observation`
2. **`with_psf`** — attach PSF to the observation
3. **`ForwardModel`** — combine lens + source + grid + PSF + mask
4. **`render` + `masked_logp`** — evaluate model image and log-likelihood
5. **`lens_mh_multistart`** — adaptive Metropolis-Hastings with multi-start
6. **`chain_stats`** — posterior mean and std after burn-in

The noise model (`GaussPoissNoise`) flows automatically from `obs.noise`
into `masked_logp` — no manual sigma needed.